
# One-call fit helper overview

The one-call helpers ``fit23``, ``fit24``, ``fit25``, and ``fit26`` wrap the
reusable fit classes. They are useful for notebooks and scripts where one decay
or one pattern mixture is fitted at a time. This example runs every helper and
then runs the matching class API with the same data.

Use the class APIs when many decays share the same IRF, background, and
instrument settings.


In [ ]:
import numpy as np

import tttrlib


def make_jordi_irf(n_channels=32, period=32.0):
    time_axis = np.linspace(0.0, period, n_channels * 2)
    irf = (
        np.exp(-0.5 * ((time_axis - 2.0) / 0.25) ** 2)
        + np.exp(-0.5 * ((time_axis - 18.0) / 0.25) ** 2)
    )
    return irf.astype(np.float64), time_axis


irf, time_axis = make_jordi_irf()
background = np.zeros_like(irf)
dt = time_axis[1] - time_axis[0]
period = 32.0

low_count_decay = np.array(
    [
        0, 0, 0, 1, 9, 7, 5, 5, 5, 2, 2, 0, 0, 0, 0, 0,
        1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 3, 2, 2, 2, 2, 3, 0, 1, 0, 1, 1, 1, 2,
        0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
    ],
    dtype=np.int32,
)

helper23 = tttrlib.fit23(
    data=low_count_decay,
    irf=irf,
    background=background,
    dt=dt,
    period=period,
    tau=2.2,
    gamma=0.01,
    r0=0.38,
    rho=1.2,
)

fit24_background = np.zeros_like(irf) + 0.2
fit24_model = np.zeros_like(irf)
tttrlib.DecayFit24.modelf(
    np.array([4.0, 0.01, 0.5, 0.9, 1.0]),
    irf,
    fit24_background,
    dt,
    np.array([period, 1.0, 0.1, 0.1, len(irf) // 2 - 1]),
    fit24_model,
)
fit24_data = np.random.default_rng(0).poisson(
    fit24_model * 200_000 / fit24_model.sum()
)
helper24 = tttrlib.fit24(
    data=fit24_data,
    irf=irf,
    background=fit24_background,
    dt=dt,
    period=period,
    tau1=3.5,
    gamma=0.02,
    tau2=0.7,
    a2=0.5,
    offset=1.0,
)

helper25 = tttrlib.fit25(
    data=low_count_decay,
    irf=irf,
    background=background,
    dt=dt,
    period=period,
    taus=(0.5, 1.0, 2.0, 4.0),
    gamma=0.02,
    r0=0.38,
    fit_gamma=False,
)

pattern_1 = irf + 0.001
pattern_2 = np.exp(-np.linspace(0.0, 6.0, len(irf))) + 0.001
true_fraction_1 = 0.35
fit26_data = np.random.default_rng(1).poisson(
    (
        true_fraction_1 * pattern_1 / pattern_1.sum()
        + (1.0 - true_fraction_1) * pattern_2 / pattern_2.sum()
    )
    * 5_000
)
helper26 = tttrlib.fit26(
    data=fit26_data,
    pattern_1=pattern_1,
    pattern_2=pattern_2,
    fraction_1=0.5,
)

class23 = tttrlib.Fit23(
    dt=dt,
    irf=irf,
    background=background,
    period=period,
    g_factor=1.0,
    l1=0.1,
    l2=0.1,
    convolution_stop=len(irf) // 2 - 1,
)
class_result23 = class23(
    data=low_count_decay,
    initial_values=np.array([2.2, 0.01, 0.38, 1.2]),
    fixed=np.array([0, 1, 1, 1], dtype=np.int16),
)

class24 = tttrlib.Fit24(
    dt=dt,
    irf=irf,
    background=fit24_background,
    period=period,
    convolution_stop=len(irf) // 2 - 1,
)
class_result24 = class24(
    data=fit24_data,
    initial_values=np.array([3.5, 0.02, 0.7, 0.5, 1.0]),
    fixed=np.array([0, 0, 0, 0, 1], dtype=np.int16),
)

class25 = tttrlib.Fit25(
    dt=dt,
    irf=irf,
    background=background,
    period=period,
    g_factor=1.0,
    l1=0.1,
    l2=0.1,
    convolution_stop=len(irf) // 2 - 1,
)
class_result25 = class25(
    data=low_count_decay,
    initial_values=np.array([0.5, 1.0, 2.0, 4.0, 0.02, 0.38]),
    fixed=np.array([1, 1, 1, 1, 1, 1], dtype=np.int16),
)

class26 = tttrlib.Fit26(pattern_1=pattern_1, pattern_2=pattern_2)
class_result26 = class26(
    data=fit26_data,
    initial_values=np.array([0.5, 0.0]),
    fixed=np.array([0], dtype=np.int16),
)

print("Helper vs class results")
print("=======================")
print(f"fit23 tau helper/class: {helper23['x'][0]:.3f} / {class_result23['x'][0]:.3f}")
print(
    "fit24 tau1/tau2 helper/class: "
    f"{helper24['x'][0]:.3f}/{helper24['x'][2]:.3f} / "
    f"{class_result24['x'][0]:.3f}/{class_result24['x'][2]:.3f}"
)
print(f"fit25 selected tau helper/class: {helper25['x'][0]:.3f} / {class_result25['x'][0]:.3f}")
print(f"fit26 fraction_1 helper/class: {helper26['x'][0]:.3f} / {class_result26['x'][0]:.3f}")